## Preprocessing Config

In [ ]:
CONFIG_PATH = "../Pipeline_semtab/Preprocessing/config/config_generated.txt"

INPUT_FOLDER = "WikidataTables2024R1/DataSets/Valid/tables"
OUTPUT_FOLDER = "WikidataTables2024R1/DataSets/Valid/preprocessing_nollm"
TARGET_FOLDER = "WikidataTables2024R1/DataSets/Valid/targets"
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
NEED_CORRECTTYPO = False
NUM_GPUS = 4
SYSTEM_PROMPT = "You are a spelling corrector. Correct only obvious typos in the text the user sends. Do not translate, do not rephrase, do not change proper nouns, casing, numbers or dates. If the text is already correct, return it unchanged. Reply with the corrected text only, with no explanation, no quotes and no prefixes."
USE_FEWSHOT = True
#Very important the format of the fewshot example entity -> entity 
FEWSHOT_EXAMPLES = "pariss -> paris; United Kngdom -> United Kingdom; 1999 -> 1999; UnipolSai -> UnipolSai" 

In [ ]:
import os

def generate_preprocessing_config(path=None):
    path = path or CONFIG_PATH
    entries = {
        "INPUT_FOLDER": INPUT_FOLDER,
        "OUTPUT_FOLDER": OUTPUT_FOLDER,
        "TARGET_FOLDER": TARGET_FOLDER,
        "MODEL_NAME": MODEL_NAME,
        "NEED_CORRECTTYPO": NEED_CORRECTTYPO,
        "NUM_GPUS": NUM_GPUS,
        "SYSTEM_PROMPT": SYSTEM_PROMPT,
        "USE_FEWSHOT": USE_FEWSHOT,
        "FEWSHOT_EXAMPLES": FEWSHOT_EXAMPLES,
    }
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for key, value in entries.items():
            value = str(value).replace("\n", "\\n")
            f.write(f"{key}:{value}\n")
    print(f"Config written: {path}")

generate_preprocessing_config("config_preprocessing.txt")

## Candidate Retrieval Config

In [ ]:
CONFIG_PATH = "../Pipeline_semtab/Candidate_Retrieval/config/config_generated.txt"

INPUT_FOLDER = "WikidataTables2024R1/DataSets/Valid/preprocessing_nollm"
OUTPUT_FOLDER = "WikidataTables2024R1/DataSets/Valid/candidate_finetuning"

# Query
GENERATOR_ORDER = "direct,llm"
USE_DIRECT = True
USE_LLM = True
USE_FUZZY = False


# Wikidata
LANGUAGE = "en"
SEARCH_LIMIT = 10
MAX_CANDIDATES_PER_CELL = 0
API_SLEEP = 0.1
ENRICH_CANDIDATES = True

# LLM
MODEL_NAME = "zai-org/glm-4-9b-chat-hf"
ADAPTER_PATH = None 
LOAD_IN_4BIT = False
LLM_MAX_SUGGESTIONS = 8
LLM_MAX_NEW_TOKENS = 1024
PROMPT = "You are a candidate generation assistant for entity linking in tabular data. Given a cell value and its table context, generate a list of distinct Wikidata entity names that could match the cell value. Return ONLY the candidate names separated by semicolons, nothing else. Example output: Candidate1; Candidate2; Candidate3"

In [ ]:
import os

def generate_candidate_config(path=None):
    path = path or CONFIG_PATH
    entries = {
        "INPUT_FOLDER": INPUT_FOLDER,
        "OUTPUT_FOLDER": OUTPUT_FOLDER,
        "GENERATOR_ORDER": GENERATOR_ORDER,
        "USE_DIRECT": USE_DIRECT,
        "USE_LLM": USE_LLM,
        "USE_FUZZY": USE_FUZZY,
        "LANGUAGE": LANGUAGE,
        "SEARCH_LIMIT": SEARCH_LIMIT,
        "MAX_CANDIDATES_PER_CELL": MAX_CANDIDATES_PER_CELL,
        "API_SLEEP": API_SLEEP,
        "ENRICH_CANDIDATES": ENRICH_CANDIDATES,
        "MODEL_NAME": MODEL_NAME,
        "ADAPTER_PATH": ADAPTER_PATH,
        "LOAD_IN_4BIT": LOAD_IN_4BIT,
        "LLM_MAX_SUGGESTIONS": LLM_MAX_SUGGESTIONS,
        "LLM_MAX_NEW_TOKENS": LLM_MAX_NEW_TOKENS,
        "PROMPT": PROMPT
    }
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for key, value in entries.items():
            if value is None: 
                continue
            value = str(value).replace("\n", "\\n")
            f.write(f"{key}:{value}\n")
    print(f"Config written: {path}")

generate_candidate_config()

## Ranking Config

In [ ]:
CONFIG_PATH = "../Pipeline_semtab/Ranking/config/config_generated.txt"

INPUT_FOLDER = "WikidataTables2024R1/DataSets/Valid/candidate_finetuning"
PREPROCESS_FOLDER = "WikidataTables2024R1/DataSets/Valid/preprocessing_nollm"
OUTPUT_FOLDER = "results/wikidata2024/ranking_llm"

# Method: full_slm, limited_slm , slm_context
METHOD = "slm"
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER_PATH = None   
LOAD_IN_4BIT = False
SYSTEM_PROMPT = "You are an expert assistant for entity linking in tabular data against Wikidata. Follow the output format exactly."

# Which tasks to produce (comma-separated): cea,cta,cpa
TASKS = "cea,cta,cpa"

# CEA scoring: weights "string_sim,quality,type_coherence"
CEA_WEIGHTS = "0.5,0.2,0.3"
# Number of top candidates sent to the SLM on a CEA tie-break (method heuristic)
CEA_LLM_TOPK = 5

# Small-LM involvement
CEA_USE_SLM = True
CTA_USE_SLM = True
CPA_USE_SLM = True
CEA_TIEBREAK_MARGIN = 0.05
CEA_CONTEXT_TIEBREAK = False
CEA_CONTEXT_MARGIN = 0.10

# CTA
CTA_FROM_SELECTION = True
CTA_MARGIN = 0.3   # margin between top-2 type supports below which the SLM is asked
CTA_TOPK = 5       # number of candidate types sent to the SLM

# Method slm_context (gate: uncertain|all)
LLM_GATE = "uncertain"
LLM_CONTEXT_MARGIN = 0.10
LLM_CONTEXT_MAX_ROWS = 10
LLM_ENRICH = False
CONTEXT_COT = False

# Self-consistency (method llm_context)
CONTEXT_SELF_CONSISTENCY = False
SC_SAMPLES = 5
SC_TEMPERATURE = 0.7
SC_TOP_P = 0.95

# Method llm (debate + verify)
LLM_VERIFY = True
LLM_TOPK = 5

# Engine
DEVICE_ID = 0
MAX_CTX = 8192
LLM_BATCH_SIZE = 32

# Wikidata
LANGUAGE = "en"    
API_SLEEP = 0.1   

# Output format
ENTITY_AS_URI = True
WRITE_HEADER = False
ROW_OFFSET = 1  # +1 for the offset


In [ ]:
CEA_PROMPT = (
    "Cell value: ${mention}\n${header_block}${row_block}\n"
    "Candidate entities:\n${candidates}\n\n"
    "Return only the QID of the entity that best matches the cell "
    "value in this context."
)

CONTEXT_PROMPT = (
    "You are disambiguating one cell of a table against Wikidata.\n\n"
    "${table_block}Target cell (${location}) value: ${mention}\n"
    "${header_block}${coltype_block}\n"
    "Candidate entities:\n${candidates}\n\n${instruction}"
)

CONTEXT_INSTRUCTION_ENRICH = (
    "The other columns of the marked row (>>...<<) describe the "
    "SAME entity (e.g. a place, date, category or related value). "
    "Compare those cells against each candidate's description, "
    "aliases and type, and pick the candidate they fit. Return "
    "ONLY the QID of the best match."
)

CONTEXT_INSTRUCTION_PLAIN = (
    "Using the whole table as context (the other columns and rows "
    "describe the same kind of thing), return only the QID of the "
    "entity that best matches the target cell."
)

CONTEXT_INSTRUCTION_COT = (
    "Reason step by step: weigh the target cell value, its column header, "
    "likely column type and the row context against each candidate's "
    "label, description and type. Briefly say why the close alternatives "
    "are eliminated, then finish with one single final line, exactly in "
    "the form:\nAnswer: <QID>"
)

CTA_PROMPT = (
    "${header_block}${values_block}\n"
    "Candidate types:\n${candidates}\n\n"
    "Return only the QID of the type that best describes all the "
    "values in this column."
)

CPA_PROMPT = (
    "${header_block}${values_block}\n"
    "Candidate properties (subject -> object):\n${candidates}\n\n"
    "Return only the PID of the property that best links the two columns."
)

DEBATE_PROMPT = (
    "Cell value: ${mention}\n${header_block}${row_block}\n"
    "Candidate entities from Wikidata:\n${candidates}\n\n"
    "Select the best matching entity and give 3 short arguments.\n"
    "Output format:\nQID: <qid>\nArguments: <arguments>"
)

VERIFY_PROMPT = (
    "Cell value: ${mention}\n${header_block}${row_block}"
    "Currently selected entity: ${chosen}\n\n"
    "All candidates:\n${candidates}\n\n"
    "Check the selection fits the cell value, column and row context. "
    "Revise if a better candidate exists, or answer NIL if none fits.\n"
    "Output format:\nWinning QID: <qid or NIL>"
)

MAX_NEW_TOKENS_CEA = 128
MAX_NEW_TOKENS_CONTEXT = 512
MAX_NEW_TOKENS_CONTEXT_COT = 512
MAX_NEW_TOKENS_CTA = 128
MAX_NEW_TOKENS_CPA = 128
MAX_NEW_TOKENS_DEBATE = 256
MAX_NEW_TOKENS_VERIFY = 256

In [ ]:
import os

def generate_ranking_config(path=None):
    path = path or CONFIG_PATH
    entries = {
        "INPUT_FOLDER": INPUT_FOLDER,
        "PREPROCESS_FOLDER": PREPROCESS_FOLDER,
        "OUTPUT_FOLDER": OUTPUT_FOLDER,
        "METHOD": METHOD,
        "MODEL_NAME": MODEL_NAME,
        "ADAPTER_PATH": ADAPTER_PATH,
        "LOAD_IN_4BIT": LOAD_IN_4BIT,
        "SYSTEM_PROMPT": SYSTEM_PROMPT,
        "TASKS": TASKS,
        "CEA_WEIGHTS": CEA_WEIGHTS,
        "CEA_LLM_TOPK": CEA_LLM_TOPK,
        "CEA_USE_SLM": CEA_USE_SLM,
        "CTA_USE_SLM": CTA_USE_SLM,
        "CPA_USE_SLM": CPA_USE_SLM,
        "CEA_TIEBREAK_MARGIN": CEA_TIEBREAK_MARGIN,
        "CEA_CONTEXT_TIEBREAK": CEA_CONTEXT_TIEBREAK,
        "CEA_CONTEXT_MARGIN": CEA_CONTEXT_MARGIN,
        "CTA_FROM_SELECTION": CTA_FROM_SELECTION,
        "CTA_MARGIN": CTA_MARGIN,
        "CTA_TOPK": CTA_TOPK,
        "LLM_GATE": LLM_GATE,
        "LLM_CONTEXT_MARGIN": LLM_CONTEXT_MARGIN,
        "LLM_CONTEXT_MAX_ROWS": LLM_CONTEXT_MAX_ROWS,
        "LLM_ENRICH": LLM_ENRICH,
        "CONTEXT_COT": CONTEXT_COT,
        "CONTEXT_SELF_CONSISTENCY": CONTEXT_SELF_CONSISTENCY,
        "SC_SAMPLES": SC_SAMPLES,
        "SC_TEMPERATURE": SC_TEMPERATURE,
        "SC_TOP_P": SC_TOP_P,
        "LLM_VERIFY": LLM_VERIFY,
        "LLM_TOPK": LLM_TOPK,
        "DEVICE_ID": DEVICE_ID,
        "MAX_CTX": MAX_CTX,
        "LLM_BATCH_SIZE": LLM_BATCH_SIZE,
        "LANGUAGE": LANGUAGE,
        "API_SLEEP": API_SLEEP,
        "ENTITY_AS_URI": ENTITY_AS_URI,
        "WRITE_HEADER": WRITE_HEADER,
        "ROW_OFFSET": ROW_OFFSET,
        "CEA_PROMPT": CEA_PROMPT,
        "CONTEXT_PROMPT": CONTEXT_PROMPT,
        "CONTEXT_INSTRUCTION_ENRICH": CONTEXT_INSTRUCTION_ENRICH,
        "CONTEXT_INSTRUCTION_PLAIN": CONTEXT_INSTRUCTION_PLAIN,
        "CONTEXT_INSTRUCTION_COT": CONTEXT_INSTRUCTION_COT,
        "CTA_PROMPT": CTA_PROMPT,
        "CPA_PROMPT": CPA_PROMPT,
        "DEBATE_PROMPT": DEBATE_PROMPT,
        "VERIFY_PROMPT": VERIFY_PROMPT,
        "MAX_NEW_TOKENS_CEA": MAX_NEW_TOKENS_CEA,
        "MAX_NEW_TOKENS_CONTEXT": MAX_NEW_TOKENS_CONTEXT,
        "MAX_NEW_TOKENS_CONTEXT_COT": MAX_NEW_TOKENS_CONTEXT_COT,
        "MAX_NEW_TOKENS_CTA": MAX_NEW_TOKENS_CTA,
        "MAX_NEW_TOKENS_CPA": MAX_NEW_TOKENS_CPA,
        "MAX_NEW_TOKENS_DEBATE": MAX_NEW_TOKENS_DEBATE,
        "MAX_NEW_TOKENS_VERIFY": MAX_NEW_TOKENS_VERIFY,
    }
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for key, value in entries.items():
            if value is None: 
                continue
            value = str(value).replace("\n", "\\n")
            f.write(f"{key}:{value}\n")
    print(f"Config written: {path}")

generate_ranking_config()